In [ ]:
import pandas as pd
import torch
import re
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
import torch
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

---

**data cleansing** 

In [ ]:
def remove_numeric(text):
    return re.sub(r'\d+', '', text)


# define the regular expression pattern to match the label and text
pattern = r"^(\d)\t(.+)$"

# create an empty list to store the data
data = []

# open the text file and iterate through each line
with open("VideoCommentsThreatCorpus.txt", "r") as file:
    for line in file:
        # remove any leading/trailing white spaces
        line = line.strip()
        # check if the line matches the pattern
        match = re.match(pattern, line)
        if match:
            # extract the label and text from the matched line
            label = int(match.group(1))
            text = match.group(2)
            # append the label and text to the data list
            data.append([text, label])

# create a Pandas dataframe from the data list

df = pd.DataFrame(data, columns=["text", "label"])

df["text"] = df["text"].apply(lambda x: remove_numeric(x))

In [ ]:
df = df.drop_duplicates()

def remove_special_characters(text):
    pattern = r'[^a-zA-Z0-9\s]'
    text = re.sub(pattern, '', text)
    return text

df['text'] = df['text'].apply(remove_special_characters)

df['text'] = df['text'].apply(lambda x: x.lower())

**determine max_sequence_length of your data**

In [ ]:
seq_len = [len(row.split()) for row in df["text"]]
pd.Series(seq_len).hist(bins=50)

In [ ]:
max_length = 40

indexes = []

for i, row in df.iterrows():

    if len(row["text"].split()) > max_length:

        indexes.append(i)

df.drop(indexes, inplace=True)

stemming and lemmatization

In [ ]:
nltk.download("punkt")
nltk.download("wordnet")

# Instantiate the stemmer and lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Define a function to apply stemming and lemmatization to a sentence
def stem_and_lemmatize(sentence):
    # Tokenize the sentence into words
    words = nltk.word_tokenize(sentence.lower())
    # Apply stemming to each word
    stemmed_words = [stemmer.stem(word) for word in words]
    # Apply lemmatization to each word
    lemmatized_words = [lemmatizer.lemmatize(word) for word in stemmed_words]
    # Join the lemmatized words back into a sentence
    lemmatized_sentence = " ".join(lemmatized_words)
    return lemmatized_sentence


# Apply the stem_and_lemmatize function to your text data
df["text"] = df["text"].apply(stem_and_lemmatize)

---

**statisticall information of dataset**

In [ ]:
threaten_comment    = df[df["label"] == 1]
un_threaten_comment = df[df["label"] == 0]

In [ ]:
print(f"size of the dataset : {df.shape[0]} ")
print(f"size of threaten comments : {threaten_comment.shape[0] } ")
print(f"size of un_threaten comments : {un_threaten_comment.shape[0] } ")

In [ ]:
print("threaten comments")
threaten_comment.head()

In [ ]:
print("un-threaten comments")
un_threaten_comment.head()

---

**sampling methods**

Stratified Sampling

In [ ]:
from sklearn.model_selection import train_test_split
# Stratified sampling
X = df["text"]
y = df["label"]
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, stratify=y)

In [ ]:
train_y.value_counts()[0] , train_y.value_counts()[1]

In [ ]:
test_y.value_counts()[0] , test_y.value_counts()[1]

balanced sampling : Sample as much minority data as majority data

In [ ]:
balanced_unthreaten = un_threaten_comment.sample(n = threaten_comment.shape[0] , random_state=42)

In [ ]:
balanced_unthreaten

**we will use the balanced sampling method**

---

train test split

In [ ]:
dataset = pd.concat([balanced_unthreaten, threaten_comment])
dataset

In [ ]:
X = dataset["text"]
y = dataset["label"]
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, stratify=y)

In [ ]:
print(f"value count of train_y \n{train_y.value_counts()} ")
print(f"value count of test_y \n{test_y.value_counts()} ")

---

In [ ]:
train_df = pd.concat([train_X, train_y], axis=1)
test_df  = pd.concat([test_X, test_y], axis=1)

In [ ]:
print(f"train_data.shape  : {train_df.shape}")

train_df.head()

In [ ]:
print(f"test_data.shape  : {test_df.shape}")

test_df.head()

---

get the tf-idf matrix of dataset

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd


vectorizer = TfidfVectorizer(
    strip_accents='unicode',
    ngram_range=(1, 2),
    min_df=0.001,
    max_df=0.8)

X_train_tf = vectorizer.fit_transform(train_df["text"])
X_test_tf = vectorizer.transform(test_df["text"])

X_train_tf

**dimension reduction using svd on tf_idf matrix**

In [ ]:
from sklearn.decomposition import TruncatedSVD

In [ ]:
truncated_svd = TruncatedSVD(n_components=200, random_state=0)
X_train_reduced = truncated_svd.fit_transform(X_train)
X_test_reduced = truncated_svd.transform(X_test)
X_train_reduced.shape, X_test_reduced.shape

In [ ]:
truncated_svd.explained_variance_ratio_.sum()

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report



In [ ]:
clf = MultinomialNB()
clf.fit(X_train, train_y)

# y_pred_train = clf.predict(X_train)
y_pred_test = clf.predict(X_test)
print(classification_report(test_y, y_pred_test))

---

**get the word representations using pretrained bert model**

In [ ]:
! pip install transformers

In [ ]:
from transformers import BertTokenizer, BertModel

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

# Define a function to get the CLS representation of a text using BERT
def get_cls_representation(text):

    tokens = tokenizer.batch_encode_plus(
        text,
        add_special_tokens=True,
        truncation=True,
        padding="max_length",
        max_length=max_length + 2,
        return_tensors="pt",
        return_attention_mask=True,
    )

    input_ids = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]

    outputs = model(input_ids, attention_mask=attention_mask)
    cls_representation = outputs[0][:, 0, :]

    return cls_representation.detach().numpy()


train_text_representations = [
    get_cls_representation(train_df["text"][i : i + 64].tolist())
    for i in range(0, train_df.shape[0], 64)
]

test_text_representations = [
    get_cls_representation(test_df["text"][i : i + 64].tolist())
    for i in range(0, test_df.shape[0], 64)
]

train_rep = np.concatenate(train_text_representations).reshape((train_df.shape[0], 768))
test_rep = np.concatenate(test_text_representations).reshape((test_df.shape[0], 768))

train_df["text_represantion"] = train_rep.tolist()

test_df["text_represantion"] = test_rep.tolist()

In [ ]:
train_df.head()

In [ ]:
test_df.head()